In [28]:
import mlflow
import mlflow.sklearn
import dagshub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from imblearn.over_sampling import RandomOverSampler
sns.set(style='whitegrid')

import warnings
warnings.filterwarnings("ignore")

In [29]:
df = pd.read_csv("C:\\projects\\MLOPS-fullstack\\ref_folder\\data.csv")
df.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,3,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,4,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,5,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


* id: Unique ID for the customer
* Gender: Gender of the customer
* Age: Age of the customer
* Driving_License: [0 : Customer does not have DL, 1 : Customer already has DL]
* Region_Code: Unique code for the region of the customer
* Previously_Insured: [1 : Customer already has Vehicle Insurance, 0 : Customer doesn't have Vehicle Insurance]
* Vehicle_Age: Age of the Vehicle
* Vehicle_Damage: [1 : Customer got his/her vehicle damaged in the past. 0 : Customer didn't get his/her vehicle damaged in the past.]
* Annual_Premium: The amount customer needs to pay as premium in the year
* Policy_Sales_Channel: Anonymized Code for the channel of outreaching to the customer ie. Different Agents, Over Mail, Over Phone, In Person, etc.
* Vintage: Number of Days, Customer has been associated with the company
* Response: [1 : Customer is interested, 0 : Customer is not interested]

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381109 entries, 0 to 381108
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    381109 non-null  int64  
 1   Gender                381109 non-null  object 
 2   Age                   381109 non-null  int64  
 3   Driving_License       381109 non-null  int64  
 4   Region_Code           381109 non-null  float64
 5   Previously_Insured    381109 non-null  int64  
 6   Vehicle_Age           381109 non-null  object 
 7   Vehicle_Damage        381109 non-null  object 
 8   Annual_Premium        381109 non-null  float64
 9   Policy_Sales_Channel  381109 non-null  float64
 10  Vintage               381109 non-null  int64  
 11  Response              381109 non-null  int64  
dtypes: float64(3), int64(6), object(3)
memory usage: 34.9+ MB


In [31]:
# Checking the target distribution 
print('Original class distribution:')
print(df['Response'].value_counts(normalize=True))

# Performing Stratified Sampling (10% of data)
X = df.drop('Response', axis=1)
y = df['Response']

# train_test_split for a stratified hold-out. 
X_sample, _, y_sample, _ = train_test_split(
    X, y,
    train_size=0.1,          
    random_state=42,      
    stratify=y              
)

# Combining back into a single DataFrame for sample
df_sample = pd.concat([X_sample, y_sample], axis=1)
print(f'\nSampled dataset shape: {df_sample.shape}')
print('Sampled class distribution:')
print(df_sample['Response'].value_counts(normalize=True))

Original class distribution:
Response
0    0.877437
1    0.122563
Name: proportion, dtype: float64

Sampled dataset shape: (38110, 12)
Sampled class distribution:
Response
0    0.877434
1    0.122566
Name: proportion, dtype: float64


In [32]:
df_sample.shape

(38110, 12)

In [33]:
# checking for null values
df_sample.isnull().sum()

id                      0
Gender                  0
Age                     0
Driving_License         0
Region_Code             0
Previously_Insured      0
Vehicle_Age             0
Vehicle_Damage          0
Annual_Premium          0
Policy_Sales_Channel    0
Vintage                 0
Response                0
dtype: int64

In [34]:
df_sample.describe()

,id,Age,Driving_License,Region_Code,Previously_Insured,Annual_Premium,Policy_Sales_Channel,Vintage,Response
count,38110.000000,38110.000000,38110.000000,38110.000000,38110.000000,38110.000000,38110.000000,38110.000000,38110.000000
mean,191145.689976,38.835161,0.997927,26.338231,0.456940,30450.433482,112.202519,154.740698,0.122566
std,110199.646422,15.522027,0.045483,13.243016,0.498149,16833.200136,54.123557,83.524551,0.327943
min,5.000000,20.000000,0.000000,0.000000,0.000000,2630.000000,1.000000,10.000000,0.000000
25%,96059.750000,25.000000,1.000000,15.000000,0.000000,24259.500000,29.000000,82.000000,0.000000
50%,191443.500000,36.000000,1.000000,28.000000,0.000000,31619.000000,136.000000,155.000000,0.000000
75%,287265.250000,50.000000,1.000000,35.000000,1.000000,39384.750000,152.000000,227.000000,0.000000
max,381086.000000,85.000000,1.000000,52.000000,1.000000,346982.000000,163.000000,299.000000,1.000000


In [35]:
# checking distribution for target column
df_sample['Response'].value_counts()

Response
0    33439
1     4671
Name: count, dtype: int64

In [36]:
num_feat = ['Age','Vintage']
cat_feat = ['Gender', 'Driving_License', 'Previously_Insured', 'Vehicle_Age_lt_1_Year',
'Vehicle_Age_gt_2_Years','Vehicle_Damage_Yes','Region_Code','Policy_Sales_Channel']

In [37]:
df_sample['Gender'] = df_sample['Gender'].map({'Female': 0, 'Male': 1})
df_sample.head(2)

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
115711,115712,1,43,1,46.0,0,1-2 Year,Yes,51672.0,11.0,38,1
334515,334516,1,24,1,50.0,0,< 1 Year,Yes,45236.0,152.0,109,0


In [38]:
for col in df.columns:
    print(f"{col} >> {df_sample[col].dtype}")

id >> int64
Gender >> int64
Age >> int64
Driving_License >> int64
Region_Code >> float64
Previously_Insured >> int64
Vehicle_Age >> object
Vehicle_Damage >> object
Annual_Premium >> float64
Policy_Sales_Channel >> float64
Vintage >> int64
Response >> int64


In [39]:
# creating dummy cols for categorical features

df_sample=pd.get_dummies(df_sample,drop_first=True)
df_sample.head(2)

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Annual_Premium,Policy_Sales_Channel,Vintage,Response,Vehicle_Age_< 1 Year,Vehicle_Age_> 2 Years,Vehicle_Damage_Yes
115711,115712,1,43,1,46.0,0,51672.0,11.0,38,1,False,False,True
334515,334516,1,24,1,50.0,0,45236.0,152.0,109,0,True,False,True


In [40]:
for col in df_sample.columns:
    print(f"{col} >> {df_sample[col].dtype}")

id >> int64
Gender >> int64
Age >> int64
Driving_License >> int64
Region_Code >> float64
Previously_Insured >> int64
Annual_Premium >> float64
Policy_Sales_Channel >> float64
Vintage >> int64
Response >> int64
Vehicle_Age_< 1 Year >> bool
Vehicle_Age_> 2 Years >> bool
Vehicle_Damage_Yes >> bool


In [41]:
# cols renaming and keeping dtype as int

df_sample = df_sample.rename(columns={"Vehicle_Age_< 1 Year": "Vehicle_Age_lt_1_Year", "Vehicle_Age_> 2 Years": "Vehicle_Age_gt_2_Years"})
df_sample['Vehicle_Age_lt_1_Year'] = df_sample['Vehicle_Age_lt_1_Year'].astype('int')
df_sample['Vehicle_Age_gt_2_Years'] = df_sample['Vehicle_Age_gt_2_Years'].astype('int')
df_sample['Vehicle_Damage_Yes'] = df_sample['Vehicle_Damage_Yes'].astype('int')

In [42]:
# scaling the data

from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler

ss = StandardScaler()
df_sample[num_feat] = ss.fit_transform(df_sample[num_feat])


mm = MinMaxScaler()
df_sample[['Annual_Premium']] = mm.fit_transform(df_sample[['Annual_Premium']])

# also, dropping id col now
id=df_sample.id
df_sample=df_sample.drop('id',axis=1)

In [43]:
# train-test split
from sklearn.model_selection import train_test_split

train_target=df_sample['Response']
train=df_sample.drop(['Response'], axis = 1)
x_train,x_test,y_train,y_test = train_test_split(train,train_target, random_state = 0)

In [44]:
train_target

115711    1
334515    0
336126    0
231948    0
260249    0
         ..
27570     0
80782     1
361828    0
338487    0
133003    0
Name: Response, Length: 38110, dtype: int64

In [45]:
train.head(1)

,Gender,Age,Driving_License,Region_Code,Previously_Insured,Annual_Premium,Policy_Sales_Channel,Vintage,Vehicle_Age_lt_1_Year,Vehicle_Age_gt_2_Years,Vehicle_Damage_Yes
115711,1,0.268321,1,46.0,0,0.142418,11.0,-1.3977,0,0,1


In [46]:
diddy

NameError: name 'diddy' is not defined

In [48]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow')
dagshub.init(repo_owner='MANJESH-ctrl', repo_name='MLOPS_proj', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logisti_Regression_Baseline")

2026-01-11 11:28:10,895 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/MANJESH-ctrl/MLOPS_proj "HTTP/1.1 200 OK"


Initialized MLflow to track repo "MANJESH-ctrl/MLOPS_proj"

2026-01-11 11:28:10,899 - INFO - Initialized MLflow to track repo "MANJESH-ctrl/MLOPS_proj"


Repository MANJESH-ctrl/MLOPS_proj initialized!

2026-01-11 11:28:10,899 - INFO - Repository MANJESH-ctrl/MLOPS_proj initialized!
2026/01/11 11:28:11 INFO mlflow.tracking.fluent: Experiment with name 'Logisti_Regression_Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/ae24899b3db84723af83784d27cdd1ac', creation_time=1768111092529, experiment_id='1', last_update_time=1768111092529, lifecycle_stage='active', name='Logisti_Regression_Baseline', tags={}>

In [ ]:
import logging
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run(run_name="lor_without_classweight"):
    start_time = time.time()

    try:
        logging.info("Training Logistic Regression model...")
        # mlflow.log_param("data_sample", 0.1)
        # mlflow.log_param("num_features", 100)
        # mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(random_state=42, class_weight=None, max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(x_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(x_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")


        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)








2026-01-11 11:28:23,428 - INFO - Starting MLflow run...
2026-01-11 11:28:24,178 - INFO - Training Logistic Regression model...
2026-01-11 11:28:24,187 - INFO - Initializing Logistic Regression model...
2026-01-11 11:28:24,187 - INFO - Fitting the model...
2026-01-11 11:28:26,389 - INFO - Model training complete.
2026-01-11 11:28:26,392 - INFO - Logging model parameters...
2026-01-11 11:28:27,019 - INFO - Making predictions...
2026-01-11 11:28:27,035 - INFO - Calculating evaluation metrics...
2026-01-11 11:28:27,084 - INFO - Logging evaluation metrics...
2026-01-11 11:28:33,675 - INFO - Saving and logging the model...
2026/01/11 11:28:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/11 11:28:51 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026-01-11 11:28:55,836 - INFO - Model training and logging completed in

🏃 View run unequaled-ape-996 at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/1/runs/dea7bcb25f4b4d5e81bf09d2fcbf0543
🧪 View experiment at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/1


In [51]:
import logging
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run(run_name="lor_with_classweight"):
    start_time = time.time()

    try:
        logging.info("Training Logistic Regression model...")
        # mlflow.log_param("vectorizer", "Bag of Words")
        # mlflow.log_param("num_features", 100)
        # mlflow.log_param("test_size", 0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(random_state=42, class_weight="balanced", max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(x_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(x_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        class_report = classification_report(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)
        

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")


        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)








2026-01-11 11:43:35,208 - INFO - Starting MLflow run...
2026-01-11 11:43:37,329 - INFO - Training Logistic Regression model...
2026-01-11 11:43:37,331 - INFO - Initializing Logistic Regression model...
2026-01-11 11:43:37,331 - INFO - Fitting the model...
2026-01-11 11:43:39,969 - INFO - Model training complete.
2026-01-11 11:43:39,969 - INFO - Logging model parameters...
2026-01-11 11:43:40,474 - INFO - Making predictions...
2026-01-11 11:43:40,483 - INFO - Calculating evaluation metrics...
2026-01-11 11:43:40,528 - INFO - Logging evaluation metrics...
2026-01-11 11:43:46,613 - INFO - Saving and logging the model...
2026/01/11 11:43:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/11 11:44:01 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026-01-11 11:44:05,925 - INFO - Model training and logging completed in

🏃 View run lor_with_classweight at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/1/runs/e45fe8815a32433b94bb422cb4e2666a
🧪 View experiment at: https://dagshub.com/MANJESH-ctrl/MLOPS_proj.mlflow/#/experiments/1
